# MOHIM stem-wise motif threshold diagnostics

`songs` 음원을 source 단위로 분리하고, 앞 30초의 모든 4마디 후보 점수를 threshold 적용 없이 확인합니다.

## 0. Drive와 motif 브랜치 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'motif'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

## 1. 실험 경로와 범위

In [ ]:
MAX_SONGS = 10
DATA_SOURCE = 'local_dataset'  # 'songs' 또는 'local_dataset'
DEVICE = 'cuda'
MOTIF_BARS = 4
MOTIF_SEARCH_SECONDS = 30.0

SONGS_DIR = Path('/content/drive/MyDrive/MOHIM/songs')
LOCAL_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/genius_pop_dataset')
DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/MOHIM/motif_stem_diagnostics')
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')
AUDIO_EXTENSIONS = {'.aac', '.flac', '.m4a', '.mp3', '.ogg', '.wav', '.webm'}

assert DATA_SOURCE in {'songs', 'local_dataset'}
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

## 1-1. `songs` 폴더 입력

`DATA_SOURCE = 'songs'`일 때만 실행되며 파일명을 곡 제목으로 사용합니다.

In [ ]:
if DATA_SOURCE == 'songs':
    song_paths = sorted(
        path for path in SONGS_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )[:MAX_SONGS]
    songs = [
        {
            'track_id': path.stem,
            'artist': '',
            'title': path.stem,
            'audio_path': path,
        }
        for path in song_paths
    ]

## 1-2. local dataset 입력

`DATA_SOURCE = 'local_dataset'`일 때 `tracks.json`과 `audio` 폴더에서 실제 음원이 있는 곡을 최대 10개 가져옵니다.

In [ ]:
if DATA_SOURCE == 'local_dataset':
    from mohim.local_dataset import index_audio_files, load_local_tracks, resolve_audio_path

    tracks_json = LOCAL_DATASET_DIR / 'tracks.json'
    audio_dir = LOCAL_DATASET_DIR / 'audio'
    assert tracks_json.is_file(), f'tracks.json이 없습니다: {tracks_json}'
    assert audio_dir.is_dir(), f'음원 폴더가 없습니다: {audio_dir}'

    tracks = load_local_tracks(tracks_json, require_lyrics=False)
    audio_index = index_audio_files(audio_dir)
    songs = []
    for track in tracks:
        audio_path = resolve_audio_path(track, audio_index)
        if audio_path is None:
            print(f'[skip] 음원 없음: {track.artist} - {track.title}')
            continue
        songs.append({
            'track_id': track.track_id,
            'artist': track.artist,
            'title': track.title,
            'audio_path': audio_path,
        })
        if len(songs) >= MAX_SONGS:
            break
    song_paths = [song['audio_path'] for song in songs]

## 1-3. 선택한 입력 확인

In [ ]:
assert songs, f'사용 가능한 음원이 없습니다: {DATA_SOURCE}'
assert len(songs) == len(song_paths)
print(f'data source: {DATA_SOURCE}')
print('songs:', len(songs))
for song in songs:
    label = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print('-', label)

## 2. Demucs와 motif scorer 준비

In [ ]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt', BEAT_CHECKPOINT
    )
separator = StemSeparator(device=DEVICE, model_name='htdemucs_6s')
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_scorer = MotifExtractor(
    beat_tracker,
    MotifConfig(bars=MOTIF_BARS, search_seconds=MOTIF_SEARCH_SECONDS),
)
print('separator and motif scorer ready')

## 3. 모든 stem × start-downbeat 후보 계산

`active_ratio >= 0.8`이고 onset/chroma similarity 차이가 0.4 이하인 모든 후보를 저장합니다.

In [ ]:
import json
import pandas as pd
import re
from mohim.separator import save_audio

def safe_folder_name(value):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' .')
    return cleaned[:120] or 'untitled'

def find_existing_song_dir(audio_path):
    resolved_audio = Path(audio_path).resolve()
    for existing_metadata_path in DIAGNOSTIC_DIR.glob('*/motif_scores.json'):
        try:
            existing_metadata = json.loads(
                existing_metadata_path.read_text(encoding='utf-8')
            )
            if Path(existing_metadata.get('source_audio', '')).resolve() == resolved_audio:
                return existing_metadata_path.parent
        except (OSError, ValueError, TypeError):
            continue
    return None

all_rows = []
for song_index, audio_path in enumerate(song_paths):
    song = songs[song_index]
    display_title = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print(f'[{song_index + 1}/{len(song_paths)}] {display_title}')
    song_id = f'{song_index:03d}_{safe_folder_name(display_title)}'
    song_dir = DIAGNOSTIC_DIR / song_id
    existing_song_dir = find_existing_song_dir(audio_path)
    if existing_song_dir is not None and existing_song_dir != song_dir:
        if not song_dir.exists():
            existing_song_dir.rename(song_dir)
            print(f'  [rename] {existing_song_dir.name} -> {song_dir.name}')
        else:
            song_dir = existing_song_dir
            song_id = song_dir.name
            print(f'  [warning] 대상 폴더가 이미 있어 기존 폴더 유지: {song_dir.name}')
    song_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = song_dir / 'motif_scores.json'

    existing = None
    if metadata_path.is_file() and (song_dir / 'melodic_accompaniment.flac').is_file():
        try:
            candidate_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            same_audio = (
                Path(candidate_metadata.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            candidates = candidate_metadata.get('candidates')
            candidate_files_exist = isinstance(candidates, list) and all(
                row.get('candidate_file')
                and (song_dir / row['candidate_file']).is_file()
                for row in candidate_metadata.get('candidates', [])
            )
            if same_audio and candidate_files_exist:
                existing = candidate_metadata
        except (OSError, ValueError, TypeError):
            existing = None

    if existing is not None:
        print('  [reuse] 기존 분리/후보 결과 사용')
        for row in existing['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})
        continue

    print('  [process] Demucs 분리 및 후보 계산')
    stems, sample_rate, _ = separator.separate(audio_path)
    result = motif_scorer.score_all(audio_path, stems, sample_rate)
    melodic = result['melodic_accompaniment']
    save_audio(song_dir / 'melodic_accompaniment.flac', melodic, sample_rate, audio_format='flac')

    for row_index, row in enumerate(result['candidates']):
        start = round(row['start_sec'] * sample_rate)
        end = round(row['end_sec'] * sample_rate)
        filename = f"candidate_{row_index:03d}_{row['stem_name']}.flac"
        save_audio(song_dir / filename, melodic[:, start:end], sample_rate, audio_format='flac')
        row['candidate_file'] = filename
        all_rows.append({'song_id': song_id, 'title': display_title, **row})

    metadata = {
        'song_id': song_id, 'title': display_title, 'source_audio': str(audio_path),
        'sample_rate': sample_rate, 'candidates': result['candidates'],
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    del stems, result, melodic

scores_df = pd.DataFrame(all_rows)
scores_df.to_csv(DIAGNOSTIC_DIR / 'all_motif_scores.csv', index=False)
display(scores_df.sort_values(['song_id', 'start_sec', 'stem_name']))

## 4. 곡당 비교 후보 2개 별도 저장

`onset_similarity >= 0.6` 후보에서 stem별 최초 후보를 하나씩 고른 뒤, 그중 시작 시간이 가장 빠른 후보와 similarity가 가장 높은 후보를 별도 JSON과 FLAC으로 저장합니다. 기존 `motif_scores.json`은 수정하지 않습니다.

In [ ]:
import shutil

metadata_paths = sorted(DIAGNOSTIC_DIR.glob('*/motif_scores.json'))
assert metadata_paths, '완료된 진단 결과가 없습니다.'

for metadata_path in metadata_paths:
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    for stale_name in (
        'motif_selections.json',
        'selected_earliest.flac',
        'selected_highest_similarity.flac',
    ):
        stale_path = metadata_path.parent / stale_name
        if stale_path.is_file():
            stale_path.unlink()
    eligible = [
        row for row in metadata['candidates']
        if row['onset_similarity'] >= 0.60
    ]
    first_by_stem = {}
    for row in sorted(eligible, key=lambda item: item['start_sec']):
        first_by_stem.setdefault(row['stem_name'], row)
    stem_candidates = list(first_by_stem.values())
    if not stem_candidates:
        print(f'[skip] onset 통과 후보 없음: {metadata["title"]}')
        continue

    selections = {
        'earliest': dict(min(stem_candidates, key=lambda row: row['start_sec'])),
        'highest_similarity': dict(max(stem_candidates, key=lambda row: row['similarity'])),
    }
    for label, row in selections.items():
        selection_file = f'selected_{label}.flac'
        shutil.copy2(metadata_path.parent / row['candidate_file'], metadata_path.parent / selection_file)
        row['selection_file'] = selection_file

    selection_metadata = {
        'song_id': metadata['song_id'],
        'title': metadata['title'],
        'source_audio': metadata['source_audio'],
        'onset_threshold': 0.60,
        'first_by_stem': first_by_stem,
        'selections': selections,
    }
    (metadata_path.parent / 'motif_selections.json').write_text(
        json.dumps(selection_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )

print('곡당 비교 후보 저장 완료')

## 5. 선택한 두 후보 자동 재생

곡 번호만 선택하면 earliest와 highest similarity 후보를 연속으로 표시합니다.

In [ ]:
from IPython.display import Audio, display

DEBUG_TRACK_INDEX = 0
selection_paths = sorted(DIAGNOSTIC_DIR.glob('*/motif_selections.json'))
assert selection_paths, '저장된 motif_selections.json이 없습니다.'
selection_path = selection_paths[DEBUG_TRACK_INDEX]
selection_metadata = json.loads(selection_path.read_text(encoding='utf-8'))
source_audio = Path(selection_metadata['source_audio']).resolve()
selected_song = next(
    (song for song in songs if Path(song['audio_path']).resolve() == source_audio),
    None,
)
if selected_song is None:
    print(selection_metadata['title'])
else:
    display_title = (
        f"{selected_song['artist']} - {selected_song['title']}"
        if selected_song['artist'] else selected_song['title']
    )
    print(display_title)

for label, title in (
    ('earliest', '1. Earliest start among first stem candidates'),
    ('highest_similarity', '2. Highest similarity among first stem candidates'),
):
    row = selection_metadata['selections'][label]
    print(f'\n{title}')
    print({key: value for key, value in row.items() if key not in {'candidate_file', 'selection_file'}})
    display(Audio(filename=str(selection_path.parent / row['selection_file'])))